### Bronze (ingest)

In [0]:
import logging

logging.basicConfig(level=logging.INFO)

logger = logging.getLogger(__name__)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
logger.info("Starting naijamart lakehouse pipeline")

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/my_files/naijamart/")

In [0]:
customer_schema = StructType([
  StructField("customer_id", StringType(), True),
  StructField("full_name", StringType(), True),
  StructField("email", StringType(), True),
  StructField("phone", StringType(), True),
  StructField("city", StringType(), True),
  StructField("signup_date", StringType(), True)
])
order_schema = StructType([
  StructField("order_id", StringType(), True),
  StructField("customer_id", StringType(), True),
  StructField("product_id", StringType(), True),
  StructField("quantity", StringType(), True),
  StructField("discount_pct", StringType(), True),
  StructField("order_date", StringType(), True),
  StructField("payment_method", StringType(), True),
  StructField("order_status", StringType(), True)
])
product_schema = StructType([
  StructField("category", StringType(), True),
  StructField("product_id", StringType(), True),
  StructField("product_name", StringType(), True),
  StructField("supplier", StringType(), True),
  StructField("unit_price", StringType(), True)
])

In [0]:
customers = spark.read.format("csv").schema(customer_schema).option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/default/my_files/naijamart/naijamart_customers.csv").select("*", col("_metadata.file_name").alias("_source_file"))
orders = spark.read.format("csv").option("header", "true").schema(order_schema).option("inferSchema", "true").load("/Volumes/workspace/default/my_files/naijamart/naijamart_orders.csv").select("*", col("_metadata.file_name").alias("_source_file"))
products = spark.read.format("json").option("header", "true").schema(product_schema).option("inferSchema", "true").load("/Volumes/workspace/default/my_files/naijamart/naijamart_products.json").select("*", col("_metadata.file_name").alias("_source_file"))
logger.info("DataFrame loaded successfully")

In [0]:
customers = customers.withColumn("_ingested_at", current_timestamp())
orders = orders.withColumn("_ingested_at", current_timestamp())
products = products.withColumn("_ingested_at", current_timestamp())
customers.display()
orders.display()
products.display()

In [0]:
initial_customers_rows = customers.count()
logger.info("Initial customers row count: %s", initial_customers_rows)
initial_orders_rows = orders.count()
logger.info("Initial orders row count: %s", initial_orders_rows)
initial_products_rows = products.count()
logger.info("Initial products row count: %s", initial_products_rows)

In [0]:
customers.write.mode("overwrite").saveAsTable("customers")
orders.write.mode("overwrite").saveAsTable("orders")
products.write.mode("overwrite").saveAsTable("products")

In [0]:
logger.info("Data ingested and saved as tables")

### Silver (clean, dedupe, validate)

#### Clean

In [0]:
logger.info("Beginning data cleaning")

In [0]:
string_columns = [name for name, dtype in customers.dtypes if dtype == "string"]

customers = customers.withColumns({
    name: when(trim(col(name)) == "", None).otherwise(trim(col(name)))
    for name in string_columns
})

In [0]:
string_columns = [name for name, dtype in orders.dtypes if dtype == "string"]

orders = orders.withColumns({
    name: when(trim(col(name)) == "", None).otherwise(trim(col(name)))
    for name in string_columns
})

In [0]:
string_columns = [name for name, dtype in products.dtypes if dtype == "string"]

products = products.withColumns({
    name: when(trim(col(name)) == "", None).otherwise(trim(col(name)))
    for name in string_columns
})

In [0]:
customers = customers.withColumns({
    "full_name": initcap(col("full_name")),
    "city": initcap(col("city")),
    "email": when(~col("email").contains("@"), None)
    .otherwise(lower(col("email")))
})
orders = orders.withColumns({
    "order_status": when (lower(trim(col("order_status"))) == "bank transfer", "transfer")
    .otherwise(lower(trim(col("order_status")))),
    "payment_method": lower(col("payment_method")),
    "discount_pct": when(col("discount_pct").isNull(), 0)
    .otherwise(col("discount_pct"))
})
products = products.withColumns({
    "unit_price": regexp_replace(col("unit_price"), r"[₦,]", "").cast("double"),
    "category": initcap(col("category"))
})
customers.display()
orders.display()
products.display()

In [0]:
customers = customers.withColumn(
    "signup_date",
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy"),
        try_to_date(col("signup_date"), "yyyy/MM/dd"),
        try_to_date(col("signup_date"), "yyyy-MM-dd HH:mm:ss")
    )
)
orders = orders.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "yyyy-MM-dd HH:mm:ss")
    )
)
customers.display()
orders.display()

In [0]:
logger.info("Data cleaning complete")

#### Deduplication

In [0]:
logger.info("Deduplicating data")

In [0]:
from pyspark.sql.window import Window
customers = customers.dropna(subset=["customer_id"])
customers = customers.withColumn(
    "rank",
    row_number().over(Window.partitionBy("customer_id").orderBy(col("_ingested_at").desc()))
)
deduplicated_customers = customers.filter(col("rank") == 1).drop("rank")
deduplicated_products = products.dropDuplicates(["product_id"])
deduplicated_orders = orders.dropDuplicates(["order_id"])
deduplicated_customers_count = deduplicated_customers.count()
deduplicated_orders_count = deduplicated_orders.count()
deduplicated_products_count = deduplicated_products.count()
logger.info("Customers deduplicated row count: %s", deduplicated_customers_count)
logger.info("Orders deduplicated row count: %s", deduplicated_orders_count)
logger.info("Products deduplicated row count: %s", deduplicated_products_count)

In [0]:
logger.info("Data deduplication complete")

#### Validation

In [0]:
logger.info("Beginning data validation")

In [0]:
Rejected_customers = concat_ws(",", when(col("full_name").isNull(), "Missing name"),
                               when(col("signup_date").isNull(), "Missing signup date")
)
Rejected_products = concat_ws(",", when(col("product_name").isNull(), "Missing product name"),
                              when(col("unit_price").isNull(), "Missing unit price"),
                              when(col("unit_price") < 0, "Negative unit price")
)
valid_silver_customers = customers.filter(~(Rejected_customers != ""))
valid_silver_products = products.filter(~(Rejected_products != ""))
Rejected_customers = customers.withColumn("reject_reason", Rejected_customers) \
    .filter(col("reject_reason") != "")
Rejected_products = products.withColumn("reject_reason", Rejected_products) \
    .filter(col("reject_reason") != "")
Rejected_customers_count =  Rejected_customers.count()
Rejected_products_count = Rejected_products.count()
logger.info("Rejected customers count: %s", Rejected_customers_count)
logger.info("Rejected products count: %s", Rejected_products_count)
Rejected_customers.display()
Rejected_products.display()

In [0]:
invalid_customer_orders = orders.join(
    valid_silver_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
).select("order_id").distinct()

invalid_product_orders = orders.join(
    valid_silver_products.select("product_id"),
    on="product_id",
    how="left_anti"
).select("order_id").distinct()

orders = orders.withColumn(
    "invalid_customer",
    col("order_id").isin(
        invalid_customer_orders.select("order_id")
    )
).withColumn(
    "invalid_product",
    col("order_id").isin(
        invalid_product_orders.select("order_id")
    )
)

In [0]:
Rejected_orders = concat_ws(",", when(col("quantity").isNull(), "Missing quantity"),
                            when(col("quantity") < 0, "Negative quantity"),
                            when(col("discount_pct").isNull(), "Missing discount"),
                            when(col("discount_pct") < 0, "Negative discount"),
                            when(col("order_date").isNull(), "Missing order date"),
                            when(~col("order_status").isin("delivered", "shipped", "pending", "cancelled", "returned"), \
                            "Invalid order status"),
                            when(~lower(trim(col("payment_method"))).isin("card","transfer", "bank transfer", "pos", "cash", "ussd"), \
                            "Invalid payment method"),
                            when(col("discount_pct") > 100, "Discount greater than 100%"),
                            when(col("discount_pct") < 0, "Discount less than 0%"),
                             when(col("invalid_customer"), "Customer not in valid Silver customers"),
                             when(col("invalid_product"), "Product not in valid Silver products")
)
valid_silver_orders = orders.filter(~(Rejected_orders != ""))
Rejected_orders = orders.withColumn("reject_reason", Rejected_orders) \
    .filter(col("reject_reason") != "")
Rejected_orders_count = Rejected_orders.count()
logger.info("Rejected orders count: %s", Rejected_orders_count)
Rejected_orders.display()

In [0]:
Rejected_customers.drop("rank").write.mode("overwrite").saveAsTable("Rejected_customers")
Rejected_products.write.mode("overwrite").saveAsTable("Rejected_products")
Rejected_orders.write.mode("overwrite").saveAsTable("Rejected_orders")

In [0]:
logger.info("Valid Silver customers count: %s", valid_silver_customers.count())
logger.info("Valid Silver products count: %s", valid_silver_products.count())
logger.info("Valid Silver orders count: %s", valid_silver_orders.count())

In [0]:
valid_silver_orders = valid_silver_orders.drop("invalid_customer", "invalid_product")

In [0]:
valid_silver_orders = valid_silver_orders.join(
    valid_silver_products.select("product_id", "unit_price"),
    on="product_id",
    how="inner"
)
valid_silver_orders = valid_silver_orders.withColumn("total_amount", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100))
valid_silver_orders».display()

In [0]:
logger.info("Data validation completed")

### Run Summary

In [0]:
logger.info("Building summary")

In [0]:
summary_data = [
    {
        "table_name": "customers",
        "raw_rows": customers.count(),
        "duplicates_removed": customers.count() - deduplicated_customers.count(),
        "rejected_rows": Rejected_customers.count(),
        "valid_rows": valid_silver_customers.count()
    },
    {
        "table_name": "products",
        "raw_rows": products.count(),
        "duplicates_removed": products.count() - deduplicated_products.count(),
        "rejected_rows": Rejected_products.count(),
        "valid_rows": valid_silver_products.count()
    },
    {
        "table_name": "orders",
        "raw_rows": orders.count(),
        "duplicates_removed": orders.count() - deduplicated_orders.count(),
        "rejected_rows": Rejected_orders.count(),
        "valid_rows": valid_silver_orders.count()
    }
]
pipeline_run_summary = spark.createDataFrame(summary_data)
pipeline_run_summary.display()

In [0]:
pipeline_run_summary.write.format("delta").mode("overwrite").saveAsTable("pipeline_run_summary")

In [0]:
logger.info("Ran summary")

### Gold(Spark SQL)